### Build Drivers Dimension (`dim_drivers`)

This notebook builds the **Drivers Dimension Table** in the gold layer of our Formula 1 data lakehouse. It is part of the **medallion architecture** pipeline (bronze -> silver -> gold) and sits in the `04-gold` folder.

##### What this notebook does:
1. **Reads** the `drivers` table from the silver layer and the `ref_nationalaty_regions` reference table from the gold layer
2. **Joins** them on `nationality` to enrich each driver with its geographic region
3. **Selects** only the columns needed for the dimension: driver_id, driver_name, date_of_birth, nationality, and region
4. **Writes** the result as a Delta table to `formula1.gold.dim_drivers`

##### Purpose:
The `dim_drivers` table serves as a denormalized dimension table that allows analysts and dashboards to query driver information enriched with region data - without needing to perform joins at query time.

##### Dependencies:
- **Upstream config:** **`00-common/01.environment-config`** (provides `catalog_name`, `silver_schema`, `gold_schema` variables)
- **Source tables:** `formula1.silver.drivers`, `formula1.gold.ref_nationalaty_regions`
- **Target table:** `formula1.gold.dim_drivers`

In [0]:
%run ../00-common/01.environment-config 

##### Step 0: Import Required Functions
Here we import all the built-in functions from `pyspark.sql.functions` (such as `col`, `lit`, `when`, `concat`, etc.). This gives us access to all the transformation and aggregation functions we might need when working with our DataFrames later in the notebook.

In [0]:
from pyspark.sql.functions import *

##### Define the Gold Target Table
We define the fully qualified target table name where our final dimension table will be written. The variable `target_table` is set to `formula1.gold.dim_drivers` by combining:
- `catalog_name` -> the Unity Catalog name (`formula1`)
- `gold_schema` -> the gold layer schema (`gold`)
- Table name -> `dim_drivers`

This follows the **medallion architecture** (bronze -> silver -> gold), where the gold layer holds clean, business-ready dimension and fact tables.

In [0]:

target_table = f'{catalog_name}.{gold_schema}.dim_drivers'

##### Step 1: Read the Source Tables
We read two tables into Spark DataFrames:
- `drivers_df` -> reads `formula1.silver.drivers` which contains driver details (driver_id, driver_name, date_of_birth, nationality)
- `ref_nationality_region_df` -> reads `formula1.gold.ref_nationalaty_regions` which is a reference table mapping each nationality to its geographic region (e.g., "British" -> "Europe", "Mexican" -> "North America")

Both DataFrames are loaded lazily (Spark won't actually read the data until an action like `display()` or `write` is triggered). The `nationality` column exists in both tables and will be used as the join key in the next step.

In [0]:
drivers_df = spark.table(f'{catalog_name}.{silver_schema}.drivers')
ref_nationality_region_df = spark.read.table(f'{catalog_name}.{gold_schema}.ref_nationalaty_regions')

##### Step 2: Join Drivers with Nationality-Region Mapping
We create a new DataFrame called `dim_drivers_df` by performing a **left outer join** between `drivers_df` and `ref_nationality_region_df` on the `nationality` column. This enriches each driver with its geographic region.

**Join condition:** `drivers_df.nationality == ref_nationality_region_df.nationality`

**Join type:** Left Outer - ensures ALL drivers are kept even if no matching nationality is found in the reference table (region would be `null` in that case).

**Selected columns from the joined result:**
| Column | Source DataFrame | Description |
|--------|-----------------|-------------|
| `driver_id` | drivers_df | Unique identifier for the driver |
| `driver_name` | drivers_df | Full name of the driver |
| `date_of_birth` | drivers_df | Driver's date of birth |
| `nationality` | drivers_df | Nationality of the driver (e.g., "British", "Dutch") |
| `region` | ref_nationality_region_df | Geographic region (e.g., "Europe", "Asia", "North America") |

In [0]:
dim_drivers_df =(
    drivers_df
     .join(
        ref_nationality_region_df,
        drivers_df.nationality == ref_nationality_region_df.nationality,
        'left_outer')
    .select(
        drivers_df.driver_id,
        drivers_df.driver_name,
        drivers_df.date_of_birth,
        drivers_df.nationality,
        ref_nationality_region_df.region.alias('nationality_region'))
)

##### Step 3: Write to Gold Layer
Finally, we write the `dim_drivers_df` DataFrame to the gold schema as a Delta table called `dim_drivers`. This table serves as a **dimension table** that combines driver identity with geographic region into a single, denormalized table ready for analytics and reporting.

- **Format:** Delta (supports ACID transactions, time travel, and schema evolution)
- **Mode:** SaveAsTable (creates or replaces the table)
- **Target:** `formula1.gold.dim_drivers`

In [0]:
(
    dim_drivers_df
    .write
    .mode('overwrite')
    .option('overwriteSchema', True)
    .format('delta')
    .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))

### Entity Relationship Diagram

The diagram below shows how the **source tables** relate to each other and how they combine into the **target dimension table** (gold layer).

```
┌─────────────────────────────────────┐         ┌─────────────────────────────────────────────┐
│       SILVER LAYER (Source)         │         │           GOLD LAYER (Target)               │
├─────────────────────────────────────┤         ├─────────────────────────────────────────────┤
│                                     │         │                                             │
│  ┌───────────────────────────┐      │         │  ┌───────────────────────────────────────┐  │
│  │  silver.constructors      │      │         │  │       gold.dim_constructors            │  │
│  ├───────────────────────────┤      │         │  ├───────────────────────────────────────┤  │
│  │ PK constructor_id         │      │         │  │  constructor_id   (from constructors)  │  │
│  │    constructor_name       │      │  JOIN   │  │  constructor_name (from constructors)  │  │
│  │    nationality ───────────│──┐   │ ──────► │  │  nationality      (from constructors)  │  │
│  │    ingestion_timestamp    │  │   │         │  │  region           (from ref table)     │  │
│  │    source_file            │  │   │         │  └───────────────────────────────────────┘  │
│  └───────────────────────────┘  │   │         │                                             │
│                                  │   │         │                                             │
└──────────────────────────────────┼───┘         └─────────────────────────────────────────────┘
                                   │
         ┌─────────────────────────┼─────┐
         │  GOLD LAYER (Reference) │     │
         ├─────────────────────────┼─────┤
         │                         │     │
         │  ┌──────────────────────┴──┐  │
         │  │ gold.ref_nationalaty_   │  │
         │  │ regions                 │  │
         │  ├─────────────────────────┤  │
         │  │    nationality ─────────│──┘
         │  │    region               │
         │  └─────────────────────────┘
         │                               │
         └───────────────────────────────┘
```

**Relationship:** `constructors.nationality` (FK) → `ref_nationalaty_regions.nationality` (PK) — **LEFT OUTER JOIN**

**Result:** One row per constructor, enriched with geographic region. Constructors without a matching nationality retain `null` for region.